# gentletask v7 — Example Usage

Reference implementation: [`gentletask.py`](./gentletask.py) · Spec: [`spec.md`](./spec.md)

`gentletask` solves two problems Python's stdlib concurrency leaves open:

- **Stopability** — cooperatively stop a hierarchy of running tasks; the stop
  signal propagates inward and the parent waits for its children.
- **Semantic context** — a `SemanticStack` (the `throughline` singleton) carries
  *what your program is doing* across thread and process boundaries, so logs tell
  a story instead of a list of disconnected events.

This notebook walks through both.

In [1]:
import logging
import sys
import time

sys.path.insert(0, ".")

from gentletask import (
    ThreadTask, WorkTask, WorkerThread, asynch,
    throughline, current_task, task_chain,
    ThroughlineNameFilter, Task,
    sleep, check_stop, poll, Queue, Event, Stopped,
)

## 1. `SemanticStack` / `throughline`

The `throughline` is a context-local stack of labeled frames. Each `with`
block pushes a frame of arbitrary key/value pairs; leaving the block pops it.
It has no opinion about tasks — it just holds whatever keys you give it.

In [2]:
with throughline(operation="abc123", user="alice"):
    with throughline(operation="resize"):
        print("get  innermost operation:", throughline.get("operation"))
        print("get  user (outer frame): ", throughline.get("user"))
        print("collect operation:       ", throughline.collect("operation"))
        print("frames:")
        for f in throughline.frames():
            print("   ", f)

get  innermost operation: resize
get  user (outer frame):  alice
collect operation:        ('abc123', 'resize')
frames:
    {'operation': 'abc123', 'user': 'alice'}
    {'operation': 'resize'}


`get(key)` returns the **innermost** value. `collect(key)` returns **all**
values outermost-first, skipping frames that don't have the key. `frames()`
hands back fresh dicts — mutating them can't corrupt the stack.

In [3]:
with throughline(name="acquisition"):
    with throughline(name="calibrate"):
        # walk() maps a function over each frame, outermost-first
        print("collect name:", throughline.collect("name"))
        print("walk len:    ", throughline.walk(lambda f: len(f)))

collect name: ('acquisition', 'calibrate')
walk len:     (1, 1)


### Snapshots

`snapshot()` captures the current frames; `restore()` reinstalls them for a
block. This is the machinery that carries context onto worker threads.

In [4]:
with throughline(name="request_handler", request_id="r-42"):
    snap = throughline.snapshot()

# Out here the stack is empty again...
print("outside:", throughline.collect("name"))

# ...but restore() brings the captured context back.
with snap.restore():
    print("restored:", throughline.collect("name"), throughline.get("request_id"))

outside: ()
restored: ('request_handler',) r-42


## 2. `ThreadTask` basics

`ThreadTask` runs a callable in a new daemon thread. `wait()` blocks and returns
the result (or re-raises the exception). `result` is shorthand for `wait()`.

In [5]:
t = ThreadTask(lambda: 6 * 7)
print("result:", t.wait())
print("is_done:", t.is_done)

# Exceptions surface through wait()
def boom():
    raise ValueError("kaboom")

failed = ThreadTask(boom)
try:
    failed.wait()
except ValueError as e:
    print("caught:", e)

result: 42
is_done: True
caught: kaboom


In [6]:
# args / kwargs, an explicit name, and a finish callback
log_lines = []

t = ThreadTask(
    lambda x, y: x + y,
    args=(3,), kwargs={"y": 4},
    name="adder",
    on_finish=lambda result, exc: log_lines.append(f"adder finished -> {result}"),
)
print("sum:", t.wait())
print("callback:", log_lines)

sum: 7
callback: ['adder finished -> 7']


## 3. The `asynch` factory

`asynch(fn)` returns a launcher: calling it starts a `ThreadTask`.

In [7]:
add = asynch(lambda x, y: x + y, name="async-add")
task = add(10, 20)
print("async result:", task.wait())

async result: 30


## 4. Cooperative stop

Tasks are never interrupted mid-line. The stop-aware primitives (`sleep`,
`check_stop`, `Queue.get`, `Event.wait`, `poll`) check `current_task()` at each
wait site and raise `Stopped` when a stop has been requested.

In [8]:
progress = []

def counter():
    n = 0
    while True:
        n += 1
        progress.append(n)
        sleep(0.02)   # stop-aware: raises Stopped when asked to stop

t = ThreadTask(counter, name="counter")
time.sleep(0.1)        # let it tick a few times
t.stop()               # request cooperative stop
try:
    t.wait()
except Stopped:
    print("counter stopped after", len(progress), "ticks")
print("is_stopped:", t.is_stopped)

counter stopped after 5 ticks
is_stopped: True


## 5. Stop cascades to children

When you stop a task, the stop propagates to every child task it started — and
when a task finishes it stops any still-running children (unless they were
detached).

In [9]:
child_ticks = []

def child():
    while True:
        child_ticks.append(1)
        sleep(0.02)

def parent():
    c = ThreadTask(child, name="child")
    c.wait()           # parent blocks on the child

p = ThreadTask(parent, name="parent")
time.sleep(0.1)
p.stop()               # cascades into child
try:
    p.wait()
except Stopped:
    pass
time.sleep(0.05)
print("parent stopped:", p.is_stopped)
print("child stopped too — ticks frozen at", len(child_ticks))

parent stopped: True
child stopped too — ticks frozen at 5


### `detach()` opts a child out of the cascade

A detached child keeps running after its parent stops.

In [10]:
finished = []

def long_child():
    time.sleep(0.15)
    finished.append("child done on its own")

def parent_detaching():
    c = ThreadTask(long_child, name="detached-child")
    c.detach()         # parent.stop() will NOT reach this child
    sleep(10)

p = ThreadTask(parent_detaching, name="parent")
time.sleep(0.02)
p.stop()
try:
    p.wait()
except Stopped:
    pass
time.sleep(0.2)
print(finished)

['child done on its own']


## 6. `WorkerThread` — serialized jobs with inherited context

A `WorkerThread` is a long-lived thread that runs submitted jobs one at a time.
The key detail: `submit()` snapshots the throughline **at submit time**, so the
job inherits the context of the code that *caused* it — not the worker's.

In [11]:
worker = WorkerThread(name="io-worker")

captured = {}

def job(label):
    captured[label] = task_chain()
    return label.upper()

# Submitted inside a frame -> job inherits it
with throughline(name="request_handler"):
    a = worker.submit(job, ("a",), name="job-a")
    a.wait()

# Submitted outside any frame -> job sees only its own name
b = worker.submit(job, ("b",), name="job-b")
b.wait()

print("job-a chain:", captured["a"])
print("job-b chain:", captured["b"])
print("result a:", a.result)
worker.stop()

job-a chain: ('request_handler', 'job-a')
job-b chain: ('job-b',)
result a: A


## 7. Logging integration

`ThroughlineNameFilter` injects `throughline.collect("name")` onto every log
record as `record.throughline`, giving each line its full task ancestry.

In [12]:
logger = logging.getLogger("gentletask.demo")
logger.setLevel(logging.INFO)
logger.handlers.clear()

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(throughline)s | %(message)s"))
handler.addFilter(ThroughlineNameFilter())
logger.addHandler(handler)
logger.propagate = False

def calibrate():
    logger.info("measuring offset")

def acquire():
    logger.info("starting acquisition")
    ThreadTask(calibrate, name="calibrate").wait()
    logger.info("acquisition complete")

ThreadTask(acquire, name="acquisition").wait()

('acquisition',) | starting acquisition


('acquisition', 'calibrate') | measuring offset


('acquisition',) | acquisition complete


The chain in front of each line — `('acquisition',)`,
`('acquisition', 'calibrate')` — comes straight from the throughline, across the
thread boundary, with no manual plumbing.

## 8. A custom `Task`

`Task` is a structural `Protocol` — any object with the right shape qualifies.
Push a frame with `task=self` (so `current_task()` works) and `name=...` (so the
log chain works) when your work runs.

In [13]:
class CountdownTask:
    """A minimal hand-rolled Task that counts down on a background thread."""
    def __init__(self, n):
        import threading
        self._n = n
        self.is_stopped = False
        self.is_done = False
        self._result = None
        self._done = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def _run(self):
        with throughline(name="countdown", task=self):
            try:
                while self._n > 0 and not self.is_stopped:
                    self._n -= 1
                    sleep(0.01)
                self._result = "liftoff" if not self.is_stopped else "aborted"
            finally:
                self.is_done = True
                self._done.set()

    def wait(self, timeout=None):
        self._done.wait(timeout)
        return self._result

    @property
    def result(self):
        return self.wait()

    def stop(self):
        self.is_stopped = True

    def add_finish_callback(self, fn):
        self.wait(); fn(self._result, None)

    def detach(self):
        pass

cd = CountdownTask(5)
print("is a Task:", isinstance(cd, Task))
print("result:", cd.wait())

is a Task: True
result: liftoff
